# v14 PLM — Stage 2 training on vast.ai (run as-is, top to bottom)

End result: `plm_weights.pt` ready to download.

This notebook uses a dedicated **venv** (`/workspace/venv`) and calls its
python EXPLICITLY in every cell — this sidesteps the classic Jupyter trap
where `!pip` installs into a different interpreter than the kernel
imports from (which is exactly what bit us on this image).
The venv is created with `--system-site-packages` so it inherits the
image's CUDA torch instead of re-downloading 3GB.

**Before running — on your Mac:** commit & push everything the run needs:
```
git add CommunitySolutions/chronos_solver/v14 CommunitySolutions/chronos_solver/v13/*.json
git commit -m 'v14 PLM' && git push
```

Private repo? Put a token in `GIT_TOKEN` (Cell 1). All cells use absolute
paths and re-define their own variables — safe to re-run in any order
after a kernel restart. Training runs via `nohup` (survives tab drops).

In [1]:
# ---------------- Cell 1: clone / update the repo ----------------
import os
GIT_TOKEN = ""          # only needed if the repo is private
BRANCH    = "main"      # branch that contains the v14 folder

url = (f"https://{GIT_TOKEN}@github.com/shreyasmahimkar/arc-agi-3"
       if GIT_TOKEN else "https://github.com/shreyasmahimkar/arc-agi-3")
if not os.path.exists("/workspace/arc3"):
    !git clone --depth 1 --branch {BRANCH} {url} /workspace/arc3
else:
    !cd /workspace/arc3 && git pull
!ls /workspace/arc3/CommunitySolutions/chronos_solver/v14

remote: Enumerating objects: 22, done.
remote: Counting objects: 100% (22/22), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 12 (delta 10), reused 12 (delta 10), pack-reused 0 (from 0)
Unpacking objects: 100% (12/12), 2.89 KiB | 493.00 KiB/s, done.
From https://github.com/shreyasmahimkar/arc-agi-3
   3b7e487..44e0fa5  main       -> origin/main
Updating 3b7e487..44e0fa5
Fast-forward
 .../chronos_solver/v13/v13_bfs_cache_tr87.json     |  1 +
 .../chronos_solver/v13/v13_progress.json           | 12 ++-
 CommunitySolutions/chronos_solver/v14/gen_data.py  | 12 ++-
 .../chronos_solver/v14/plm/encoder.py              | 24 ++++++
 .../chronos_solver/v14/vast-train-v14.ipynb        | 87 ++++++++++++----------
 5 files changed, 93 insertions(+), 43 deletions(-)
 create mode 100644 CommunitySolutions/chronos_solver/v13/v13_bfs_cache_tr87.json
DEPLOYMENT.md  claude-code-v14-plm.ipynb  my_agent.py  train_wm.py
README.md      gen_data.py		  plm	       vast-train-v14.ipynb


In [2]:
# ---------------- Cell 2: venv + dependencies ----------------
# --system-site-packages: inherit the image's CUDA torch (no 3GB re-pull).
# Everything from here on runs through PY — one interpreter, no ambiguity.
import os, sys
PY = "/workspace/venv/bin/python"
if not os.path.exists(PY):
    !{sys.executable} -m venv --system-site-packages /workspace/venv

W = "/workspace/arc3/arc-prize-2026-arc-agi-3/arc_agi_3_wheels"
!{PY} -m pip -q install --ignore-installed blinker
!{PY} -m pip -q install {W}/arcengine-0.9.3-py3-none-any.whl \
    {W}/arc_agi-0.9.8-py3-none-any.whl python-dotenv

# hard verification THROUGH the venv python — fails loudly if anything broke
!{PY} -c "import arcengine, arc_agi, torch; print('deps verified | torch', torch.__version__, '| cuda', torch.cuda.is_available())"


[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: /workspace/venv/bin/python -m pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.1.2
[notice] To update, run: /workspace/venv/bin/python -m pip install --upgrade pip
deps verified | torch 2.12.0+cu130 | cuda True


In [3]:
# ---------------- Cell 3: sanity — GPU + module smoke test ----------------
PY = "/workspace/venv/bin/python"
!{PY} -c "import torch; p=torch.cuda.get_device_properties(0); print(torch.cuda.get_device_name(0), f'{p.total_memory/1e9:.1f} GB')"
!cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && {PY} -m plm.smoke

/bin/bash: line 1: {PY}: command not found
tokenizer OK  (recon loss 2.775)
belief core OK
/workspace/arc3/CommunitySolutions/chronos_solver/v14/plm/world_model.py:26: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  self.tf = nn.TransformerEncoder(layer, cfg.wm_layers)
world model OK
planner OK    (explored 117, win=yes)
goose OK      (picked (6, 30, 31), err 1.00)

ALL SMOKE TESTS PASSED


In [4]:
# ---------------- Cell 4: generate training data (~5 min) ----------------
PY = "/workspace/venv/bin/python"
!cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && \
    {PY} gen_data.py --out /workspace/v14_shards \
        --episodes-per-game 400 --max-steps 150
!ls -lh /workspace/v14_shards | tail -5
!df -h /workspace | tail -1

2026-06-11 04:46:41,558 [INFO] ar25: 400 episodes (115 expert-seeded), 55637 transitions, 24 WIN events saved
2026-06-11 04:47:47,318 [INFO] bp35: 400 episodes (134 expert-seeded), 25893 transitions, 38 WIN events saved
2026-06-11 04:48:16,890 [INFO] cd82: 400 episodes (127 expert-seeded), 40380 transitions, 39 WIN events saved
2026-06-11 04:48:34,993 [INFO] cn04: 400 episodes (114 expert-seeded), 30876 transitions, 20 WIN events saved
2026-06-11 04:49:21,337 [INFO] dc22: 400 episodes (129 expert-seeded), 40847 transitions, 15 WIN events saved
2026-06-11 04:50:09,822 [INFO] ft09: 400 episodes (115 expert-seeded), 44852 transitions, 16 WIN events saved
2026-06-11 04:51:05,252 [INFO] g50t: 400 episodes (0 expert-seeded), 52000 transitions, 0 WIN events saved
2026-06-11 04:51:38,439 [INFO] ka59: 400 episodes (121 expert-seeded), 41674 transitions, 40 WIN events saved
2026-06-11 04:52:31,174 [INFO] lf52: 400 episodes (0 expert-seeded), 31712 transitions, 0 WIN events saved
2026-06-11 04:53

In [ ]:
# # ---------------- Cell 5: LAUNCH training (returns immediately) ----------------
# # Using a raw bash invocation via subprocess to cleanly detach the background process
# import subprocess

# cmd = (
#     "cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && "
#     "nohup /workspace/venv/bin/python train_wm.py --phase all --shards /workspace/v14_shards "
#     "--epochs 20 --steps-per-epoch 1000 --bsz 256 "
#     "--holdout ls20,vc33,tu93,ft09,sp80 > /workspace/train.log 2>&1 &"
# )

# # Popen runs completely asynchronously and lets Jupyter move on immediately
# subprocess.Popen(cmd, shell=True)

# import time; time.sleep(5)
# !tail -3 /workspace/train.log

2026-06-11 05:03:23,876 [INFO] device=cuda amp=True


In [ ]:
# ---------------- Cell 5: LAUNCH training (returns immediately) ----------------
import subprocess, time

# 0. make sure no older training process is still alive
subprocess.run("pkill -f train_wm.py || true", shell=True)
time.sleep(2)

# 1. launch with the perf patches:
#    --phase wm   -> resumes the checkpointed tokenizer from plm_weights.pt,
#                    pretokenizes once (~2 min), then GPU-bound epochs.
#                    (use --phase all instead if no tokenizer was trained yet)
#    --bsz 1024   -> H200/RTX-sized batches (token path can afford it)
#    --compile    -> torch.compile on the simulator (cuda only)
cmd = (
    "cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && "
    "nohup /workspace/venv/bin/python train_wm.py --phase wm "
    "--shards /workspace/v14_shards "
    "--epochs 20 --steps-per-epoch 1000 --bsz 1024 --compile "
    "--holdout ls20,vc33,tu93,ft09,sp80 > /workspace/train.log 2>&1 &"
)
subprocess.Popen(cmd, shell=True)
time.sleep(8)
!tail -5 /workspace/train.log

In [7]:
# # ---------------- Cell 5: LAUNCH training (returns immediately) ----------------
# # nohup-detached: survives tab drops and kernel restarts.
# # Gates: tokenizer pixel_acc >= 0.995, then HELDOUT_tok_acc >= 0.90.
# PY = "/workspace/venv/bin/python"
# !cd /workspace/arc3/CommunitySolutions/chronos_solver/v14 && \
#     nohup {PY} train_wm.py --phase all --shards /workspace/v14_shards \
#         --epochs 20 --steps-per-epoch 1000 --bsz 256 \
#         --holdout ls20,vc33,tu93,ft09,sp80 > /workspace/train.log 2>&1 &
# import time; time.sleep(5)
# !tail -3 /workspace/train.log

In [11]:
# ---------------- Cell 6: MONITOR (re-run me anytime) ----------------
!ps aux | grep train_wm | grep -v grep || echo '*** TRAINING NOT RUNNING (finished or crashed - check log below) ***'
print('-' * 70)
!tail -15 /workspace/train.log
print('-' * 70)
!nvidia-smi --query-gpu=utilization.gpu,memory.used --format=csv,noheader

root        2374  107  9.4 61682288 43602884 ?   Rl   05:03   0:43 /workspace/venv/bin/python train_wm.py --phase all --shards /workspace/v14_shards --epochs 20 --steps-per-epoch 1000 --bsz 256 --holdout ls20,vc33,tu93,ft09,sp80
----------------------------------------------------------------------
2026-06-11 05:03:23,876 [INFO] device=cuda amp=True
----------------------------------------------------------------------
0 %, 4 MiB


In [ ]:
# ---------------- Cell 7: VERIFY + stage for download ----------------
# (run when Cell 6 shows training finished)
PY = "/workspace/venv/bin/python"
p = "/workspace/arc3/CommunitySolutions/chronos_solver/v14/plm_weights.pt"
!{PY} -c "import torch,os; s=torch.load('{p}', map_location='cpu', weights_only=True); print('keys:', list(s)); print(f'size: {{os.path.getsize(\"{p}\")/1e6:.1f}} MB')"
!cp {p} /workspace/plm_weights.pt && cp /workspace/train.log /workspace/train_final.log
print("Download via the Jupyter file browser (/workspace): plm_weights.pt + train_final.log")
print("Then DESTROY this instance on the vast.ai console.")

## After downloading

1. Put `plm_weights.pt` into your local `CommunitySolutions/chronos_solver/v14/`
   (`*.pt` is gitignored — it travels by hand, never via git).
2. Local gate: run the held-out-game eval vs the v13 bandit baseline.
3. Stage 3: update the `v14-plm` Kaggle dataset and submit
   (see DEPLOYMENT.md).